In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
from PIL import Image
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import zarr
from collections import Counter
from scipy.stats import shapiro
import glob
from IPython.display import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [ ]:
from PIL import Image

# load image
img = Image.open("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/tissue_hires_image.png")

# load position
coords = pd.read_parquet("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/tissue_positions.parquet")

# filter spots that are in tissue
coords = coords[coords['in_tissue'] == 1]

# load part
parts = pd.read_csv('/Users/kimchaeyoung/Desktop/DP/spots_and_parts.csv')

# 4. merge position with part info
merged = coords.merge(parts, left_on='barcode', right_on='Unnamed: 0')

# scale 
fullres_width = 40000
fullres_height = 35000
img_width, img_height = img.size

scale_x = img_width / fullres_width
scale_y = img_height / fullres_height

merged['scaled_x'] = merged['pxl_col_in_fullres'] * scale_x
merged['scaled_y'] = merged['pxl_row_in_fullres'] * scale_y

# assign colors
part_colors = {0: 'red', 1: 'blue', 2: 'green', 3: 'orange'}
merged['color'] = merged['part'].map(part_colors)

#Plot
plt.figure(figsize=(6,6))
plt.imshow(img)

plt.scatter(merged['scaled_x'], merged['scaled_y'], c=merged['color'], s=5)

plt.gca().invert_yaxis()
plt.title("Visium Spots by Part")
plt.axis('off')
plt.show()


In [ ]:
methods =  ['visium_HD_2x2_cell2loc', 'visium_HD_2x2_nnls', 'visium_HD_2x2_rctd']
cell_types = ['Hepatocytes', 'Kupffercells', 'LSECs', 'Cholangiocytes', 
              'Bcells', 'Tcells', 'Mesothelialcells', 
              'PortalVeinEndothelialcells', 'CentralVeinEndothelialcells']

for method in methods:
    print(f"\n=== Summary Statistics for {method} ===")
    
    summary = []
    
    for ct in cell_types:
        zarr_path = f"/Users/kimchaeyoung/Desktop/DP/data.zarr/tables/square_016um/obsm/{method}/{ct}"
        z = zarr.open_array(zarr_path, mode='r')
        data = np.array(z[:])
        
        summary.append({
            'Cell Type': ct,
            'Mean': np.nanmean(data),     # caculcation excluding NANs
            'Median': np.nanmedian(data),
            'Max': np.nanmax(data)
        })
    
    summary_df = pd.DataFrame(summary)
    top5 = summary_df.sort_values(by='Mean', ascending=False).head(5)
    print(top5)     


In [ ]:

pairs = [
    ('visium_HD_2x2_rctd', 'visium_HD_2x2_cell2loc'),
    ('visium_HD_2x2_rctd', 'visium_HD_2x2_nnls'),
    ('visium_HD_2x2_cell2loc', 'visium_HD_2x2_nnls')
]


methods = ['visium_HD_2x2_cell2loc', 'visium_HD_2x2_nnls', 'visium_HD_2x2_rctd']
cell_types = ['Hepatocytes', 'Kupffercells', 'LSECs', 'Cholangiocytes', 
              'Bcells', 'Tcells', 'Mesothelialcells', 
              'PortalVeinEndothelialcells', 'CentralVeinEndothelialcells']

dominant_celltypes_dict = {}

for method in methods:
    proportion_list = []
    for ct in cell_types:
        zarr_path = f"/Users/kimchaeyoung/Desktop/DP/data.zarr/tables/square_016um/obsm/{method}/{ct}"
        z = zarr.open_array(zarr_path, mode='r')
        proportion_list.append(np.array(z[:]))

    proportions = np.column_stack(proportion_list)
    dominant_idx = np.argmax(proportions, axis=1)
    dominant_celltypes = [cell_types[i] for i in dominant_idx]
    
    dominant_celltypes_dict[method] = dominant_celltypes


# Sankey Subplot
fig = make_subplots(rows=1, cols=3, 
                    specs=[[{'type':'sankey'}]*3],
                    subplot_titles=[f"{m1} ➔ {m2}" for m1, m2 in pairs])

for i, (method1, method2) in enumerate(pairs):
    dominant1 = dominant_celltypes_dict[method1]
    dominant2 = dominant_celltypes_dict[method2]

    transitions = [f"{src} → {dst}" for src, dst in zip(dominant1, dominant2)]
    transition_counts = Counter(transitions)

    nodes = list(set(dominant1 + dominant2))
    node_labels = nodes + nodes
    node_map = {ct: idx for idx, ct in enumerate(nodes)}

    sources = []
    targets = []
    values = []

    for trans, count in transition_counts.items():
        if count > 30:
            src, dst = trans.split(' → ')
            sources.append(node_map[src])
            targets.append(node_map[dst] + len(nodes))
            values.append(count)

    sankey = go.Sankey(
        node=dict(
            pad=10,
            thickness=15,
            line=dict(color="black", width=0.3),
            label=node_labels
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )

    fig.add_trace(sankey, row=1, col=i+1)

fig.update_layout(title_text="Dominant Cell Type Transitions Across Methods", 
                  font_size=10,
                  width=1500, height=600)

# 저장
fig.write_html("/Users/kimchaeyoung/Desktop/DP/sankey_all_methods.html")
print("diagrams saved")


In [ ]:
from IPython.display import Image
Image(filename="/Users/kimchaeyoung/Desktop/DP/newplot.png")

In [ ]:
from PIL import Image
methods = ['visium_HD_2x2_cell2loc', 'visium_HD_2x2_nnls', 'visium_HD_2x2_rctd']
cell_types = ['Hepatocytes', 'Kupffercells', 'LSECs', 'Cholangiocytes', 
              'Bcells', 'Tcells', 'Mesothelialcells', 
              'PortalVeinEndothelialcells', 'CentralVeinEndothelialcells']

color_mapping = {
    'Hepatocytes': 'green',
    'Kupffercells': 'brown',
    'LSECs': 'purple',
    'Cholangiocytes': 'orange',
    'Bcells': 'blue',
    'Tcells': 'red',
    'Mesothelialcells': 'pink',
    'PortalVeinEndothelialcells': 'gray',
    'CentralVeinEndothelialcells': 'cyan'
}

# 좌표 & 이미지는 한번만 로드
coords = pd.read_parquet("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/tissue_positions.parquet")
tissue_spots = coords[coords['in_tissue'] == 1]
img = Image.open("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/tissue_hires_image.png")

fullres_width = 40000   # 대략 max 기준으로 추정
fullres_height = 35000

# 현재 이미지 크기
img_width, img_height = img.size

# 스케일 팩터 계산
scale_x = img_width / fullres_width
scale_y = img_height / fullres_height

# 좌표 변환
tissue_spots['scaled_x'] = tissue_spots['pxl_col_in_fullres'] * scale_x
tissue_spots['scaled_y'] = tissue_spots['pxl_row_in_fullres'] * scale_y
  

# 🔄 반복 함수
for method in methods:
    print(f"\n=== Processing {method} ===")
    
    # 1️⃣ Proportion 데이터 로드
    proportion_list = []
    for ct in cell_types:
        zarr_path = f"/Users/kimchaeyoung/Desktop/DP/data.zarr/tables/square_016um/obsm/{method}/{ct}"
        z = zarr.open_array(zarr_path, mode='r')
        proportion_list.append(np.array(z[:]))

    proportions = np.column_stack(proportion_list)

    # 2️⃣ Dominant Cell Type 계산
    dominant_idx = np.argmax(proportions, axis=1)
    dominant_celltypes = [cell_types[i] for i in dominant_idx]
    colors = [color_mapping[ct] for ct in dominant_celltypes]

    # 3️⃣ Plot
    plt.figure(figsize=(5,5))
    plt.imshow(img)

    plt.scatter(tissue_spots['scaled_x'], tissue_spots['scaled_y'], 
            c=colors, s=1)

    plt.gca().invert_yaxis()
    plt.title(f'Dominant Cell Type - {method}')

    handles = [plt.Line2D([0], [0], marker='o', color='w', label=ct, markerfacecolor=color_mapping[ct], markersize=10) for ct in color_mapping]
    plt.legend(handles=handles, title='Cell Types', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.axis('off')
    plt.show()

    # 4️⃣ 스팟 개수 출력
    count_dict = Counter(dominant_celltypes)
    print("Distribution of Dominant Cell Types")
    for ct in cell_types:
        print(f"{ct:30}: {count_dict.get(ct, 0)} spots")


In [ ]:
base_path = '/Users/kimchaeyoung/Desktop/DP'
methods = ['RCTD', 'cell2location', 'NNLS']
parts = ['part_0', 'part_1', 'part_2', 'part_3']

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)  # 2행 2열

# axes를 1D로 펼치기
axes = axes.flatten()
cell_type_colors = {
        "Hepatocytes": "#87CEEB",
        "CentralVeinEndothelialcells": "#4682B4",
        "PortalVeinEndothelialcells": "#98FB98",
        "LSECs": "#2E8B57",
        "Mesothelialcells": "#FFC0CB",
        "Kupffercells": "#FF0000",
        "Bcells": "#FFA07A",
        "Cholangiocytes": "#FF8C00",
        "Tcells": "#DDA0DD",
        "others": "#A9A9A9"
    }

for idx, part in enumerate(parts):
    avg_proportions = {}
    
    for method in methods:
        file_list = glob.glob(f"{base_path}/{method}/*{part}*")
        if not file_list:
            continue
        file_path = file_list[0]
        df = pd.read_csv(file_path, sep='\t')
        avg_proportions[method] = df.mean()
    
    if not avg_proportions:
        continue
    
    avg_df = pd.DataFrame(avg_proportions).T
    avg_df = avg_df.fillna(0)
    
    colors = [cell_type_colors.get(ct, '#333333') for ct in avg_df.columns]
    
    avg_df.plot(kind='barh', stacked=True, color=colors, width=0.5, ax=axes[idx])
    axes[idx].set_title(part, fontsize= 30)
    axes[idx].set_xlabel('Proportion', fontsize=28)
    if idx % 2 == 0:
        axes[idx].set_ylabel('Method', fontsize =25)
    else:
        axes[idx].set_ylabel('')
    axes[idx].legend().remove()
    axes[idx].tick_params(axis='y', labelsize=25)  


# 공통 Legend 추가 (오른쪽 중앙)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.05, 0.5), loc='center left', title='Cell Types', fontsize= 25)
  

plt.tight_layout()
plt.show()


In [ ]:
# 🔹 필요한 모듈들
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 🔹 기본 설정
base_path = '/Users/kimchaeyoung/Desktop/DP'
methods = ['RCTD', 'cell2location', 'NNLS']
parts = ['part_0', 'part_1', 'part_2', 'part_3']
cell_types = [
    'Hepatocytes',
    'Kupffercells',
    'LSECs',
    'Cholangiocytes',
    'Bcells',
    'Tcells',
    'Mesothelialcells',
    'PortalVeinEndothelialcells',
    'CentralVeinEndothelialcells'
]

cell_types2 = ['Hepatocytes', 'Kupffercells', 'LSECs', 'Tcells']



# ✅ 데이터 불러오기
all_data = []

for method in methods:
    for part in parts:
        pattern = f"{base_path}/{method}/*{part}*"
        file_list = glob.glob(pattern)
        for file_path in file_list:
            df = pd.read_csv(file_path, sep="\t")
            for cell_type in cell_types:
                if cell_type in df.columns:
                    for val in df[cell_type].values:
                        all_data.append({
                            'Method': method,
                            'Part': part,
                            'CellType': cell_type,
                            'Proportion': val
                        })

viz_df = pd.DataFrame(all_data)

In [ ]:
unique_pairs = viz_df[['Method', 'CellType']].drop_duplicates()

n_cols = 2
n_rows = int(np.ceil(len(unique_pairs) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for i, (method, cell_type) in enumerate(unique_pairs.values):
    subset = viz_df[(viz_df['Method'] == method) & (viz_df['CellType'] == cell_type)]
    ax = axes[i]
    sns.boxplot(data=subset, x='Part', y='Proportion', ax=ax)

    means = subset.groupby('Part')['Proportion'].mean().round(2)
    for j, part in enumerate(means.index):
        ax.text(j, means[part] + 0.01, f"{means[part]:.2f}", ha='center', va='bottom', fontsize=9)

    ax.set_title(f"{method} - {cell_type}")

# 빈 subplot 제거
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 🔹 기본 설정
base_path = '/Users/kimchaeyoung/Desktop/DP'
methods = ['RCTD', 'cell2location', 'NNLS']
parts = ['part_0', 'part_1', 'part_2', 'part_3']
cell_order = [
    'Hepatocytes',
    'Kupffercells',
    'LSECs',
    'Cholangiocytes',
    'Bcells',
    'Tcells',
    'Mesothelialcells',
    'PortalVeinEndothelialcells',
    'CentralVeinEndothelialcells'
]

# 시각화를 위한 줄바꿈 포함된 라벨
display_labels = {
    'Hepatocytes': 'Hepatocytes',
    'Kupffercells': 'Kupffer\ncells',
    'LSECs': 'LSECs',
    'Cholangiocytes': 'Cholangio-\ncytes',
    'Bcells': 'B\ncells',
    'Tcells': 'T\ncells',
    'Mesothelialcells': 'Mesothelial\ncells',
    'PortalVeinEndothelialcells': 'PortalVein-\nEndothelialcells',
    'CentralVeinEndothelialcells': 'CentralVein-\nEndothelialcells'
}

custom_palette = {
    'part_0': '#66c2a5',
    'part_1': '#fc8d62',
    'part_2': '#8da0cb',
    'part_3': '#e78ac3'
}

# ✅ 데이터 불러오기
all_data = []
for method in methods:
    for part in parts:
        pattern = f"{base_path}/{method}/*{part}*"
        file_list = glob.glob(pattern)
        for file_path in file_list:
            df = pd.read_csv(file_path, sep="\t")
            for cell_type in cell_order:
                if cell_type in df.columns:
                    for val in df[cell_type].values:
                        all_data.append({
                            'Method': method,
                            'Part': part,
                            'CellType': cell_type,
                            'Proportion': val
                        })

viz_df = pd.DataFrame(all_data)

# 📊 시각화
for method in methods:
    plt.figure(figsize=(18, 6))
    subset = viz_df[viz_df['Method'] == method]

    ax = sns.boxplot(
        data=subset,
        x='CellType',
        y='Proportion',
        hue='Part',
        order=cell_order,
        palette=custom_palette
    )

    # std 표시 (cell type별 part 평균의 std 기준 → 아래)
    part_means = subset.groupby(['CellType', 'Part'])['Proportion'].mean().reset_index()
    stds = part_means.groupby('CellType')['Proportion'].std().round(3).reset_index()
    for _, row in stds.iterrows():
        xpos = np.where(np.array(cell_order) == row['CellType'])[0][0]
        ax.text(xpos, -0.12, f"std: {row['Proportion']:.3f}", ha='center', fontsize=18, color='black')

    # x축 라벨 업데이트
    ax.set_xticklabels([display_labels[cell] for cell in cell_order], fontsize=17)

    ax.set_title(f"{method}", fontsize=22)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion", fontsize=17)
    ax.legend(title='Part', loc='upper right', fontsize=17)

    plt.ylim(-0.15, 1.08)
    plt.tight_layout()
    plt.show()


In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 🔹 기본 설정
base_path = '/Users/kimchaeyoung/Desktop/DP'
methods = ['RCTD', 'cell2location', 'NNLS']
parts = ['part_0', 'part_1', 'part_2', 'part_3']
cell_order = [
    'Hepatocytes',
    'Kupffercells',
    'LSECs',
    'Tcells',
]

# 시각화를 위한 줄바꿈 포함된 라벨
display_labels = {
    'Hepatocytes': 'Hepatocytes',
    'Kupffercells': 'Kupffercells',
    'LSECs': 'LSECs',
    'Tcells': 'Tcells',
}

custom_palette = {
    'part_0': '#66c2a5',
    'part_1': '#fc8d62',
    'part_2': '#8da0cb',
    'part_3': '#e78ac3'
}

# ✅ 데이터 불러오기
all_data = []
for method in methods:
    for part in parts:
        pattern = f"{base_path}/{method}/*{part}*"
        file_list = glob.glob(pattern)
        for file_path in file_list:
            df = pd.read_csv(file_path, sep="\t")
            for cell_type in cell_order:
                if cell_type in df.columns:
                    for val in df[cell_type].values:
                        all_data.append({
                            'Method': method,
                            'Part': part,
                            'CellType': cell_type,
                            'Proportion': val
                        })

viz_df = pd.DataFrame(all_data)

# 📊 시각화
for method in methods:
    plt.figure(figsize=(18, 6))
    subset = viz_df[viz_df['Method'] == method]

    ax = sns.boxplot(
        data=subset,
        x='CellType',
        y='Proportion',
        hue='Part',
        order=cell_order,
        palette=custom_palette
    )

    # std 표시 (cell type별 part 평균의 std 기준 → 아래)
    part_means = subset.groupby(['CellType', 'Part'])['Proportion'].mean().reset_index()
    stds = part_means.groupby('CellType')['Proportion'].std().round(3).reset_index()
    for _, row in stds.iterrows():
        xpos = np.where(np.array(cell_order) == row['CellType'])[0][0]
        ax.text(xpos, -0.12, f"std: {row['Proportion']:.3f}", ha='center', fontsize=30, color='black')

    # x축 라벨 업데이트
    ax.set_xticklabels([display_labels[cell] for cell in cell_order], fontsize=34)

    ax.set_title(f"{method}", fontsize=40)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion", fontsize=22)
    ax.legend(title='Part', loc='upper right', fontsize=17)

    plt.ylim(-0.15, 1.08)
    plt.tight_layout()
    plt.show()


In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 🔹 기본 설정
base_path = '/Users/kimchaeyoung/Desktop/DP'
methods = ['RCTD', 'cell2location', 'NNLS']
parts = ['part_0', 'part_1', 'part_2', 'part_3']
cell_order = [
    'Hepatocytes',
    'Kupffercells',
    'LSECs',
    'Cholangiocytes',
    'Bcells',
    'Tcells',
    'Mesothelialcells',
    'PortalVeinEndothelialcells',
    'CentralVeinEndothelialcells'
]

# 시각화를 위한 줄바꿈 포함된 라벨
display_labels = {
    'Hepatocytes': 'Hepatocytes',
    'Kupffercells': 'Kupffer\ncells',
    'LSECs': 'LSECs',
    'Cholangiocytes': 'Cholangio-\ncytes',
    'Bcells': 'Bcells',
    'Tcells': 'Tcells',
    'Mesothelialcells': 'Mesothelial\ncells',
    'PortalVeinEndothelialcells': 'PortalVein-\nEndothelialcells',
    'CentralVeinEndothelialcells': 'CentralVein-\nEndothelialcells'
}

custom_palette = {
    'part_0': '#66c2a5',
    'part_1': '#fc8d62',
    'part_2': '#8da0cb',
    'part_3': '#e78ac3'
}

# ✅ 데이터 불러오기
all_data = []
for method in methods:
    for part in parts:
        pattern = f"{base_path}/{method}/*{part}*"
        file_list = glob.glob(pattern)
        for file_path in file_list:
            df = pd.read_csv(file_path, sep="\t")
            for cell_type in cell_order:
                if cell_type in df.columns:
                    for val in df[cell_type].values:
                        all_data.append({
                            'Method': method,
                            'Part': part,
                            'CellType': cell_type,
                            'Proportion': val
                        })

viz_df = pd.DataFrame(all_data)

# 📊 시각화
for method in methods:
    plt.figure(figsize=(18, 6))
    subset = viz_df[viz_df['Method'] == method]

    ax = sns.boxplot(
        data=subset,
        x='CellType',
        y='Proportion',
        hue='Part',
        order=cell_order,
        palette=custom_palette
    )

    # 평균값 아래 표시 제거됨 → 대신 std 값 아래로 이동


    # std 표시 (cell type별 part 평균의 std 기준 → 아래)
    part_means = subset.groupby(['CellType', 'Part'])['Proportion'].mean().reset_index()
    stds = part_means.groupby('CellType')['Proportion'].std().round(3).reset_index()
    for _, row in stds.iterrows():
        xpos = np.where(np.array(cell_order) == row['CellType'])[0][0]
        ax.text(xpos, -0.12, f"std: {row['Proportion']:.3f}", ha='center', fontsize=18, color='black')

    # x축 라벨 업데이트
    ax.set_xticklabels([display_labels[cell] for cell in cell_order], fontsize=17)

    ax.set_title(f"{method}", fontsize=22)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion", fontsize=18)
    ax.legend(title='Part', loc='upper right', fontsize=18)

    plt.ylim(-0.15, 1.08)
    plt.tight_layout()
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

n = len(cell_types)
rows = math.ceil(n / 3)

fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 4))
axes = axes.flatten()

for idx, celltype in enumerate(cell_types):
    ax = axes[idx]
    for method in methods:
        subset = viz_df[(viz_df['CellType'] == celltype) & (viz_df['Method'] == method)]
        if not subset.empty:
            sns.kdeplot(subset['Proportion'], label=method, ax=ax)
    
    ax.set_title(f"{celltype}")
    ax.set_xlabel('Proportion')
    ax.set_ylabel('Density')
    ax.legend()

for j in range(idx+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
adata = sc.read_10x_h5("/Users/kimchaeyoung/Desktop/DP/filtered_feature_bc_matrix.h5")

if 'gene_symbols' in adata.var.columns:
    adata.var_names = adata.var['gene_symbols']

# remove duplicates
adata.var_names_make_unique()

# calculte !C metrix
sc.pp.calculate_qc_metrics(adata, inplace=True)

In [ ]:

sc.pl.violin(adata, ['n_genes_by_counts'], jitter=0.4, rotation=45)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(adata.obs["n_genes_by_counts"], bins=50, color="skyblue", edgecolor="black")
plt.axvline(x=500, color='red', linestyle='--', linewidth=1.5, label='Threshold = 500')  # threshold 선 추가
plt.title("Number of genes per spot (Visium HD)")
plt.xlabel("Number of expressed genes")
plt.ylabel("Number of spots")
plt.grid(True)
plt.legend()  # 범례 추가
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(adata.obs["total_n_UMI_per_spot"], bins=50, color="skyblue", edgecolor="black")
plt.axvline(x=500, color='red', linestyle='--', linewidth=1.5, label='Threshold = 500')  # threshold 선 추가
plt.title("UMI counts per spot (Visium HD)")
plt.xlabel("UMI Counts")
plt.ylabel("Number of Spots")
plt.grid(True)
plt.legend()  # 범례 추가
plt.show()

In [ ]:
threshold = 600
low_quality = (adata.obs["n_genes_by_counts"] < threshold).sum()
total = len(adata)

print(f"{low_quality} out of {total} spots have fewer than {threshold} expressed genes "
      f"({low_quality/total:.1%})")

x = np.sort(adata.obs["n_genes_by_counts"])
y = np.arange(1, len(x)+1) / len(x)

plt.plot(x, y)
plt.xlabel("Number of expressed genes")
plt.ylabel("Proportion of spots")
plt.title("Cumulative distribution of gene counts per spot")
plt.grid(True)
plt.show()


In [ ]:
adata = ad.read_zarr("/Users/kimchaeyoung/Desktop/DP/data.zarr/tables/square_016um")

In [ ]:
coords = pd.read_parquet("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/tissue_positions.parquet")
coords.set_index("barcode", inplace=True)

# filter spots that are only in tissue
coords = coords.loc[adata.obs_names]
tissue_mask = coords['in_tissue'] == 1
coords_in_tissue = coords[tissue_mask]

# scaling
with open("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/scalefactors_json.json") as f:
    scales = json.load(f)

scale = scales['tissue_hires_scalef']
coords_scaled = coords_in_tissue[['pxl_col_in_fullres', 'pxl_row_in_fullres']] * scale
coords_array = coords_scaled.values  # numpy 배열로 변환


In [ ]:
img = Image.open("/Users/kimchaeyoung/Desktop/DP/016/spatial/spatial/tissue_hires_image.png")


In [ ]:
def plot_usage(method_name, result):
    used = ~result.isnull().any(axis=1).values
    used_in_tissue = used[tissue_mask.values]

    total_spots = len(used)
    rejected_spots = (~used).sum()
    print(f" Total Spots: {total_spots}")
    print(f"Rejected Spots: {rejected_spots}")
    
    # extraction of non-used spots
    rejected_coords = coords_in_tissue.loc[~used_in_tissue].copy()
    rejected_coords['pxl_col_in_fullres'] = pd.to_numeric(rejected_coords['pxl_col_in_fullres'], errors='coerce') * scale
    rejected_coords['pxl_row_in_fullres'] = pd.to_numeric(rejected_coords['pxl_row_in_fullres'], errors='coerce') * scale
    rejected_coords = rejected_coords.dropna(subset=['pxl_col_in_fullres', 'pxl_row_in_fullres'])

    plt.figure(figsize=(5, 5))
    plt.imshow(img)

    # Used / Rejected scatter
    plt.scatter(coords_array[used_in_tissue, 0], coords_array[used_in_tissue, 1], 
                s=6, color='blue', label='Used Spots', alpha=0.8)
    plt.scatter(coords_array[~used_in_tissue, 0], coords_array[~used_in_tissue, 1], 
                s=6, color='red', label='Rejected Spots', alpha=0.8)

    # add density of rejected spots
    import seaborn as sns
    sns.kdeplot(
        x=rejected_coords['pxl_col_in_fullres'],
        y=rejected_coords['pxl_row_in_fullres'],
        cmap="Reds", fill=True, alpha=0.4, bw_adjust=0.7
    )

    plt.legend(loc='upper right')
    plt.title(f"{method_name} Spot Usage (Only Tissue Area)\nBlue: Used  |  Red: Rejected + Density")
    plt.axis('off')
    plt.show()



In [ ]:
plot_usage("RCTD", adata.obsm["visium_HD_2x2_rctd"])
plot_usage("NNLS", adata.obsm["visium_HD_2x2_nnls"])
plot_usage("Cell2location", adata.obsm["visium_HD_2x2_cell2loc"])


In [ ]:
def plot_usage(method_name, result):
    used = ~result.isnull().any(axis=1).values
    used_in_tissue = used[tissue_mask.values]

    total_spots = len(used)
    rejected_spots = (~used).sum()
    print(f"Total Spots: {total_spots}")
    print(f"Rejected Spots: {rejected_spots}")
    
    # rejected spot 좌표 추출
    rejected_coords = coords_in_tissue.loc[~used_in_tissue].copy()
    rejected_coords['pxl_col_in_fullres'] = pd.to_numeric(rejected_coords['pxl_col_in_fullres'], errors='coerce') * scale
    rejected_coords['pxl_row_in_fullres'] = pd.to_numeric(rejected_coords['pxl_row_in_fullres'], errors='coerce') * scale
    rejected_coords = rejected_coords.dropna(subset=['pxl_col_in_fullres', 'pxl_row_in_fullres'])

    # Spot usage 시각화
    plt.figure(figsize=(5, 5))
    plt.imshow(img)

    plt.scatter(coords_array[used_in_tissue, 0], coords_array[used_in_tissue, 1], 
                s=6, color='blue', label='Used Spots', alpha=0.8)
    plt.scatter(coords_array[~used_in_tissue, 0], coords_array[~used_in_tissue, 1], 
                s=6, color='red', label='Rejected Spots', alpha=0.8)

    import seaborn as sns
    sns.kdeplot(
        x=rejected_coords['pxl_col_in_fullres'],
        y=rejected_coords['pxl_row_in_fullres'],
        cmap="Reds", fill=True, alpha=0.4, bw_adjust=0.7
    )

    plt.legend(loc='upper right')
    plt.title(f"{method_name} Spot Usage (Only Tissue Area)\nBlue: Used  |  Red: Rejected + Density")
    plt.axis('off')
    plt.show()

    # ─────────────────────────────
    # 📊 Rejected Spot QC Histogram
    # ─────────────────────────────
    rejected_spots_index = ~used  # 전체 기준
    rejected_gene_counts = adata.obs.loc[rejected_spots_index, 'n_genes_by_counts']
    rejected_umi_counts = adata.obs.loc[rejected_spots_index, 'total_n_UMI_per_spot']

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    sns.histplot(rejected_gene_counts, bins=50, kde=False, color='red', ax=axs[0])
    axs[0].set_title('Gene Count in Rejected Spots')
    axs[0].set_xlabel('Number of Genes')
    axs[0].set_ylabel('Frequency')
    axs[0].axvline(100, color='gray')
    axs[0].legend()

    sns.histplot(rejected_umi_counts, bins=50, kde=False, color='orange', ax=axs[1])
    axs[1].set_title('UMI Count in Rejected Spots')
    axs[1].set_xlabel('UMI Count')
    axs[1].set_ylabel('Frequency')
    axs[1].axvline(100, color='gray')
    axs[1].legend()

    plt.suptitle(f"{method_name} – QC Metrics for Rejected Spots", fontsize=14)
    plt.tight_layout()
    plt.show()

plot_usage("RCTD", adata.obsm["visium_HD_2x2_rctd"])


In [ ]:
adata.obs.columns[:10]